In [ ]:
from lightkurve import DataCube, ErrorCube, DataFrame

%load_ext autoreload
%autoreload 2

In [ ]:
from astropy.io import fits
import numpy as np
import pandas as pd

hdulist = fits.open(
    "./src/lightkurve/data/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits"
)

# Prepare DataCube

In [ ]:
flux_array = hdulist[1].data["FLUX"].astype(float)
flux_err_array = hdulist[1].data["FLUX_ERR"].astype(float)
time = hdulist[1].data["TIME"].astype(float)
time_corr = hdulist[1].data["TIMECORR"].astype(float)
c0, r0 = hdulist[1].header["1CRV4P"], hdulist[1].header["2CRV4P"]
row, col = np.arange(flux_array.shape[1]) + r0, np.arange(flux_array.shape[2]) + c0
aper = flux_array.mean(axis=0) > 10000
bkg_aper = flux_array.mean(axis=0) < 4000

time_mask = hdulist[1].data["QUALITY"] == 0

In [ ]:
flux = DataCube(
    flux_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

flux_err = ErrorCube(
    flux_err_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

In [ ]:
flux, flux_err

In [ ]:
flux

In [ ]:
flux_err

# Tests

## Downsample

In [ ]:
flux.downsample(5)

In [ ]:
flux_err.downsample(5)

## Imposing an aperture mask where `mean`(flux of pixel) > 10,000

In [ ]:
flux[:, aper]

## Sum of flux in aperture, per cadence

In [ ]:
flux[:, aper].sum(axis=1)

## Sum of flux error in aperture, per cadence. 

In [ ]:
flux_err[:, aper].sum(axis=1)

# Check types when transformed

In [ ]:
type(flux), type(flux_err)

In [ ]:
type(flux[:, aper]), type(flux_err[:, aper])

In [ ]:
type(flux[:, 0, :]), type(flux_err[:, 0, :])
type(flux[:, :, 0]), type(flux_err[:, :, 0])

In [ ]:
type(flux[:, 0, 0]), type(flux_err[:, 0, 0])

In [ ]:
type(flux[:, aper].sum(axis=0)), type(flux_err[:, aper].sum(axis=0))
type(flux[:, aper].sum(axis=1)), type(flux_err[:, aper].sum(axis=1))

# Plot the light curve

In [ ]:
import matplotlib.pyplot as plt

time = np.asarray(flux.btjd)
bkg = flux[:, bkg_aper].mean(axis=1)
bkg_err = flux_err[:, bkg_aper].mean(axis=1)

bkg -= bkg.median()

lc = flux[:, aper].sum(axis=1)
lc_err = flux_err[:, aper].sum(axis=1)

plt.figure()
plt.title("Raw")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(.88e6, .94e6)

lc = flux[:, aper].sum(axis=1) - (bkg * aper.sum())
lc_err = flux_err[:, aper].sum(axis=1) + (bkg_err * aper.sum())

plt.figure()
plt.title("Background Subtracted")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(0.88e6, 0.94e6)

## Info

repo: https://github.com/tessgi/secret/tree/christina-sketch1/src/lightkurve

# Milestones;

- ✅ Break the Cube, Frame, and Series classes into their own modules (cube.py, frame.py, series.py)
- ✅ Convert these demo functions into tests inside the package to fully test the functionality as users of lightkurve would need it. As you find any tests (breaking or passing) that cover functionality users need, add them to the tests.
    - ✅ To do this you will need to choose a reasonable piece of test data we can ship with the package. Crucially this must be small (ideally kb). You can trim down some tesscut data using lightkurve to add to the package
    - ✅ Confirm adjusted data product is appropriate and sufficient for tests
    - ✅ Trim to 50 frames
    - ✅ Put in tests/data
- ✅ Take a look at the pandas documentation and see if we can safely quiet the user warning that we have for adding our convenience functions (e.g. `flux.cadence`)
    - 🗒️ Add convenience functions to _metadata namespace
    - ❗️ Don't add existing attributes to this namespace, it messes up pandas at the core level somehow
- Currently there is an aggregate method for time. Create a similar aggregate method that will instead aggregate on columns so that;
    - ✅ DataCube.spatial_aggregate(...) -> returns a lower resolution datacube. e.g. sums pixels in row and column direction to create a lower resolution image
    - ✅ DataFrame.spatial_aggregate(...) -> returns a lower resolution dataframe
    - ✅ DataSeries has no spatial axis to aggregate
    - Create functions for ErrorCube and ErrorFrame
    - ✅ downsample methods more similar to time downsample
    - Think about binning to specific time/space points
- ✅ Tests for ErrorCubes
- Documentation
- ✅ Cube `__repr__`
- Consider bleed columns, can these be accounted for easily in the DataCube framework?

- We'll come back to; can we have in lightkurve3 a way to take cubes, frames, series and convert to fits imagehdu, tablehdu, and columnhdu.

- See what happens if I give too many rows and columns
- Change header -> "meta" or something like that
- meta could return a dataclass that is indexable, could hold relevant fits file information when loaded in
- Folding - add index. Return a new object (deepcopy) unless inplace=True
- Check downsample error
- Think about how lightkurve will interact with these objects
- Think about data quality handling
- 

## Smaller test data

In [ ]:
from lksearch import TESSSearch
from astropy.coordinates import SkyCoord

search_input = (84.291190, -80.469170)
search_input = SkyCoord(*search_input, unit="deg", frame="icrs")
search_result = TESSSearch(search_input, hlsp=False)

# You can filter two different ways to get the same result.
# search_result.filter_table(pipeline='TESScut')
downloads_table = search_result.tesscut.filter_table(sector=1).download(TESScut_size=6)
hdulist = fits.open(downloads_table["Local Path"][0])

In [ ]:
hdulist = fits.open(
    "/Users/dkgiles/.lksearch/cache/mastDownload/TESSCut/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits"
)

In [ ]:
hdulist[1].data = hdulist[1].data[:50]

In [ ]:
hdulist.writeto(
    "./src/lightkurve/data/tess-s0001-4-2_84.291190_-80.469170_6x6_astrocut.fits",
)

## Spatial aggregation

In [ ]:
flux

In [ ]:
flux_lowres = flux.spatial_aggregate(3, 3)
flux_lowres

#### change to 
`flux_lowres = flux.spatial_aggregate(factor=int)`

In [ ]:
flux_lowres = flux.spatial_aggregate(4, 4)
flux_lowres

In [ ]:
def single_cadence_frame(datacube, cadence):
    return DataFrame(
        datacube.to_array()[cadence],
        index=datacube.row[:: datacube.nrow],
        columns=datacube.column[: datacube.ncol],
    )

In [ ]:
single_cadence_frame(flux_lowres, 0)

In [ ]:
flux.downsample?

In [ ]:
flux.downsample(5)

In [ ]:
df0 = single_cadence_frame(flux, 0)
df0

In [ ]:
print(f"Sum of first 2x2: {df0.iloc[:2, :2].sum().sum()}")
df0.iloc[:2, :2]

In [ ]:
df0.spatial_aggregate(3, 3)

### Spatial downsampling like the time downsampling

In [ ]:
flux.spatial_downsample(2)

In [ ]:
flux.spatial_downsample((2, 3))

In [ ]:
row = flux.pixel_row

In [ ]:
dr = np.median(np.diff(np.sort(np.unique(row)))) * 3
bins_row = np.arange(row.min(), row.max() + dr, dr)

In [ ]:
bins_row

In [ ]:
bin_edges_left_row = pd.cut(np.sort(row), bins_row, right=False)

In [ ]:
bin_edges_left_row

In [ ]:
flux.sum().sum()

In [ ]:
flux.spatial_downsample(2).sum().sum()

In [ ]:
flux[:, :-1].spatial_downsample(2)

In [ ]:
flux[:, :-2].sum().sum()

In [ ]:
flux[:, :-1].spatial_downsample(2).sum().sum()

In [ ]:
flux[:, :, :-1].spatial_downsample(2)

In [ ]:
flux[:, :, :-2].sum().sum()

In [ ]:
flux[:, :, :-1].spatial_downsample(2).sum().sum()

In [ ]:
flux.spatial_downsample(2).to_array()[0]

In [ ]:
def single_cadence_frame(datacube, cadence):
    indices = [f"{i[0]} {i[1]}" for i in zip(datacube.index.names, datacube.index[0])]
    str_index = "; ".join(indices)
    return pd.DataFrame(
        datacube.to_array()[cadence],
        index=pd.Series(datacube.row[:: datacube.nrow], name="row"),
        columns=pd.Series(datacube.column[: datacube.ncol], name="column"),
    ).style.set_caption(str_index)

In [ ]:
f0 = single_cadence_frame(flux, 0)
f0

In [ ]:
f0.style.set_caption("name")

In [ ]:
f0.index.name = "row"

In [ ]:
f0.columns.name = "column"

In [ ]:
f0

In [ ]:
[f"{i[0]}, {i[1]}" for i in zip(flux.index.names, flux.index[0])]

In [ ]:
flux.to_dataframe(0, 0)

In [ ]:
flux.index[0]

In [ ]:
c = 0

In [ ]:
with np.printoptions(edgeitems=2, threshold=5):
    print(flux)
    for c in [0, -1]:
        indices = [
            (flux.index.names[i], flux.index[c][i])
            for i in range(len(flux.index.names))
        ]
        print(f"""{indices}\n{flux.to_array()[c]}\n...""")

In [ ]:
with pd.option_context("display.max_columns", 4):
    print(flux)
    for c in [0, -1]:
        indices = [
            (flux.index.names[i], flux.index[c][i])
            for i in range(len(flux.index.names))
        ]
        print(f"""{indices}\n{single_cadence_frame(flux, c)}\n...""")

In [ ]:
pd.set_option("display.max.columns", 4)

In [ ]:
print(repr(single_cadence_frame(flux, 0)))

In [ ]:
print(flux._repr_html_())

# Dummy data

In [ ]:
ntime, nrow, ncol = 200, 10, 14
test_data = np.ones((ntime, nrow, ncol))
row, col = np.arange(test_data.shape[1]), np.arange(test_data.shape[2])
df = DataCube(test_data, row_indices={"row": row}, col_indices={"column": col})
df_err = ErrorCube(test_data, row_indices={"row": row}, col_indices={"column": col})

In [ ]:
df, df_err

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df_err

In [ ]:
timedownsample = df.downsample(2)

In [ ]:
(df.downsample(2).to_array() == 2).all()

In [ ]:
(df.spatial_downsample(3) == 9).all().all()

In [ ]:
df.spatial_downsample((5, 7))

In [ ]:
df.spatial_downsample(3)

In [ ]:
df_err.spatial_downsample(3)

In [ ]:
(df.downsample(3).to_array() == 3).all()

In [ ]:
(df.spatial_aggregate(5, 7).to_array().round() == 4).all()

In [ ]:
(df.spatial_downsample(2).to_array() == 4).all()

In [ ]:
(
    df[:, :, :-1].spatial_downsample(2).to_array()
    == df[:, :, :-2].spatial_downsample(2).to_array()
).all()

In [ ]:
(df_err.downsample(4).to_array() == 2).all()

In [ ]:
(df_err.downsample(9).to_array() == 3).all()

In [ ]:
(df_err.to_array()[0][0] ** 2).sum()

In [ ]:
df_err

In [ ]:
df_err.spatial_aggregate(5, 7)

In [ ]:
assert (df_err.spatial_aggregate(5, 7).to_array().round() == 2).all()

In [ ]:
flux[:, [1, 2, 3], [1, 2, 3]]

# Folding

In [ ]:
flux.fold(0.3, inplace=True)
flux